In [14]:
!git clone https://github.com/Ish2905/Comemo-Dataset.git

Cloning into 'Comemo-Dataset'...
remote: Enumerating objects: 77, done.
remote: Counting objects: 100% (77/77), done.
remote: Compressing objects: 100% (57/57), done.
remote: Total 77 (delta 42), reused 44 (delta 16), pack-reused 0 (from 0)
Receiving objects: 100% (77/77), 16.26 KiB | 4.06 MiB/s, done.
Resolving deltas: 100% (42/42), done.


In [15]:
%cd /content/Comemo-Dataset

/content/Comemo-Dataset


In [16]:
!git pull

remote: Enumerating objects: 7, done.
remote: Counting objects: 100% (7/7), done.
remote: Compressing objects: 100% (1/1), done.
remote: Total 4 (delta 3), reused 4 (delta 3), pack-reused 0 (from 0)
Unpacking objects: 100% (4/4), 431 bytes | 431.00 KiB/s, done.
From https://github.com/Ish2905/Comemo-Dataset
   ce32fa0..63f865b  main       -> origin/main
Updating ce32fa0..63f865b
Fast-forward
 sql/02_slim_tables.sql | 10 +++-------
 1 file changed, 3 insertions(+), 7 deletions(-)


In [9]:
# ==== DO NOT MODIFY THIS CELL ====
from google.colab import drive
drive.mount('/content/drive')

import duckdb
import os

DB_PATH = "/content/drive/MyDrive/Capstone/comemo.db"

# HARD FAIL if Drive is not mounted
assert os.path.exists("/content/drive/MyDrive"), "Drive not mounted!"

# HARD FAIL if DB file missing (after first creation)
if not os.path.exists(DB_PATH):
    print("⚠️ comemo.db not found yet (first run only)")
else:
    print("✅ Using existing database:", DB_PATH)

con = duckdb.connect(DB_PATH)

# Sanity check
print(con.execute("SHOW TABLES").fetchdf())
# =================================


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ Using existing database: /content/drive/MyDrive/Capstone/comemo.db
          name
0  reviews_raw


In [ ]:
con.execute("""
SELECT
  'reviews_raw' AS table,
  COUNT(*) AS rows
FROM reviews_raw
UNION ALL
SELECT
  'metadata_raw',
  COUNT(*)
FROM metadata_raw
""").fetchdf()


,table,rows
0,reviews_raw,66033346
1,metadata_raw,7218481


In [18]:
con.execute("SHOW TABLES").fetchdf()



,name
0,metadata_raw


In [32]:
con.execute("""
CREATE TABLE metadata_raw AS
SELECT
    parent_asin,
    main_category,
    title,
    features,
    description,
    details,
    categories,
    store,
    price,
    TRY_CAST(average_rating AS DOUBLE) AS average_rating,
    rating_number
FROM read_json(
    '/content/drive/MyDrive/Capstone/comemo_data/metadata.jsonl',
    format = 'newline_delimited',
    sample_size = -1, -- Changed from 0 to -1 to read all input for schema detection
    columns = {
        parent_asin: 'VARCHAR',
        main_category: 'VARCHAR',
        title: 'VARCHAR',
        features: 'JSON',
        description: 'JSON',
        details: 'JSON',
        categories: 'JSON',
        store: 'VARCHAR',
        price: 'VARCHAR',
        average_rating: 'VARCHAR',
        rating_number: 'BIGINT'
    },
    ignore_errors = true
)
WHERE parent_asin IS NOT NULL;
""")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

In [33]:
con.execute("DESC metadata_raw").fetchdf()

,column_name,column_type,null,key,default,extra
0,parent_asin,VARCHAR,YES,None,None,None
1,main_category,VARCHAR,YES,None,None,None
2,title,VARCHAR,YES,None,None,None
3,features,JSON,YES,None,None,None
4,description,JSON,YES,None,None,None
5,details,JSON,YES,None,None,None
6,categories,JSON,YES,None,None,None
7,store,VARCHAR,YES,None,None,None
8,price,VARCHAR,YES,None,None,None
9,average_rating,DOUBLE,YES,None,None,None


In [34]:
con.execute("""
CREATE TABLE reviews_raw AS
SELECT
    parent_asin,
    rating,
    title,
    text,
    timestamp,
    verified_purchase,
    helpful_vote
FROM read_json(
    '/content/drive/MyDrive/Capstone/comemo_data/reviews.jsonl',
    format = 'newline_delimited'
)
WHERE parent_asin IS NOT NULL;
""")


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

In [35]:
con.execute("DESC reviews_raw").fetchdf()

,column_name,column_type,null,key,default,extra
0,parent_asin,VARCHAR,YES,None,None,None
1,rating,DOUBLE,YES,None,None,None
2,title,VARCHAR,YES,None,None,None
3,text,VARCHAR,YES,None,None,None
4,timestamp,BIGINT,YES,None,None,None
5,verified_purchase,BOOLEAN,YES,None,None,None
6,helpful_vote,BIGINT,YES,None,None,None


In [ ]:
con.close()
print("DuckDB connection closed. Tables should be persisted to /content/drive/MyDrive/Capstone/comemo.db")

In [36]:
con.execute("""
COPY reviews_raw
TO '/content/drive/MyDrive/Capstone/reviews_raw.parquet'
(FORMAT PARQUET);

COPY metadata_raw
TO '/content/drive/MyDrive/Capstone/metadata_raw.parquet'
(FORMAT PARQUET);
""")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

In [37]:
con.execute("SHOW TABLES;").fetchdf()

,name
0,metadata_raw
1,reviews_raw
